<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 5 精华 — Full Multi-Agent Research System

**一句话定位**:**把前 4 节学的零件全部拼起来** —— Scoping + Research Agent + Supervisor + Report Generation,完整的 deep research pipeline。**这一节学的是「**怎么组装**」,不是新概念**。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fde047;}</style>

**🎁 这节是整门课的「**集大成**」**

前 4 节我们一块一块搭建:

| 节 | 搭好的零件 |
|----|-----------|
| **Lesson 1** | Scoping(澄清 + brief 生成) |
| **Lesson 2** | Single Research Agent(ReAct + 压缩) |
| **Lesson 3** | MCP 集成(可选,提供工具) |
| **Lesson 4** | Supervisor(派单多 agent) |
| **Lesson 5(本节)** | **Final Report Generation + 完整组装** |

→ 学完这节,你就有了 **生产级 deep research 系统**(类似 OpenAI Deep Research、Perplexity Pro Search 的架构核心)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🏗️ 完整 Pipeline — 4 大阶段

</div>

```
用户请求
  │
  ▼
┌──────────────────────────────────────┐
│ 🅰️ Scoping 阶段                       │
│ ├─ clarify_with_user                  │ ← Lesson 1
│ └─ write_research_brief               │
└──────────┬───────────────────────────┘
           │
           ▼  research_brief
┌──────────────────────────────────────┐
│ 🅱️ Supervisor 阶段                    │
│ ├─ supervisor LLM 决定派单            │ ← Lesson 4
│ ├─ ConductResearch(子主题)           │
│ │     ↓                               │
│ │   research_agent(隔离 context)     │ ← Lesson 2 / 3
│ │     ↓                               │
│ │   compressed_research               │
│ └─ think_tool(supervisor 反思)        │
└──────────┬───────────────────────────┘
           │
           ▼  notes(所有子研究的总结)
┌──────────────────────────────────────┐
│ 🅲 Report Generation 阶段             │
│ ├─ final_report_generation 节点       │ ← 本节新增
│ │   读 notes + research_brief         │
│ │   ↓                                  │
│ │   LLM 写综合性报告                  │
│ └─ 输出 final_report                  │
└──────────┬───────────────────────────┘
           │
           ▼
        END(给用户的报告)
```

| 阶段 | 来自 | 输入 | 输出 |
|------|------|------|------|
| **Scoping** | Lesson 1 | 用户对话 | `research_brief` |
| **Research(via Supervisor)** | Lesson 2 / 3 / 4 | brief | `notes`(综合 findings) |
| **Report** | 本节 | brief + notes | `final_report`(给用户) |

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧩 「子图作 node 嵌入」的递归组合性

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># 直接复用前面 notebook 里 %%writefile 保存的零件
from deep_research_from_scratch.research_agent_scope import scope_research
from deep_research_from_scratch.multi_agent_supervisor import supervisor_agent
from deep_research_from_scratch.research_agent_full import final_report_generation

deep_researcher_builder = StateGraph(AgentState, input_schema=AgentInputState)

# 把已编译的子图 当作 node 嵌进来
<span style="background:#3d3414; color:#fde047; padding:0 2px;">deep_researcher_builder.add_node("scope_research", scope_research)</span>
<span style="background:#3d3414; color:#fde047; padding:0 2px;">deep_researcher_builder.add_node("supervisor_subgraph", supervisor_agent)</span>
deep_researcher_builder.add_node("final_report_generation", final_report_generation)

deep_researcher_builder.add_edge(START, "scope_research")
deep_researcher_builder.add_edge("scope_research", "supervisor_subgraph")
deep_researcher_builder.add_edge("supervisor_subgraph", "final_report_generation")
deep_researcher_builder.add_edge("final_report_generation", END)

agent = deep_researcher_builder.compile()
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 LangGraph 的「**递归组合性**」**

**编译过的子图,本身就是个 node** —— 可以直接 `add_node("name", subgraph)` 嵌进外层图。

- `scope_research` 是个完整的 mini-workflow(2 个 node)
- `supervisor_agent` 是个完整的 mini-workflow(supervisor + 子 agent)
- 把它们 **作为黑盒** 嵌进 deep_researcher,**层级无限**

这就是为啥 LangGraph 的 **底层原语就 3 个**(State / Node / Edge),但能搭出 **任意复杂的系统**。

**类比**:像 **React 组件嵌套** —— 大组件包小组件,小组件还能包更小的,**没有层级限制**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔁 Recursion Limit — 别让循环卡死

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 LangGraph 默认 recursion limit = 25**

防止无限循环,LangGraph **默认每个图最多跑 25 步**。复杂 deep research 必须调高。

**`recursion_limit` 统计每个 node 的执行次数**(不是 graph 的层级!),所以多层嵌套图会很快累加。

</div>

### 📊 本系统的步数估算

| 阶段 | 步数 |
|------|------|
| **Scoping** | 2-3 步(clarify + brief) |
| **Single Research Agent** | 8-12 步(tool 调用 + 压缩) |
| **Multi-Agent Supervisor** | **每个子 agent 额外 8-12 步** |
| **Iterative Research** | supervisor 多轮研究填空白 |
| **Report Generation** | 1-2 步 |
| **小计**(3 个子 agent 并行) | **~50 步** |

### 🎯 调到 50

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">result = agent.invoke(
    {"messages": [HumanMessage("研究 X")]},
    config={"<span style="background:#3d3414; color:#fde047; padding:0 2px;">recursion_limit</span>": 50}
)
</code></pre>

让 supervisor 能 **多轮研究填空白**,**全面覆盖** 复杂主题。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📝 Final Report Generation — 最后一步

</div>

Supervisor 阶段产出 `notes`(所有子研究的综合 findings)。**最后一步 = 把 notes 写成给用户的报告**。

**关键设计**:

| 输入 | 用途 |
|------|------|
| `research_brief` | **原始任务描述**,LLM 知道要回答什么 |
| `notes` | **所有研究 findings** 的综合 |
| **`final_report` prompt** | 指导 LLM 怎么 **结构化呈现** |

### 🧠 Report Prompt 的关键准则

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

好报告的 4 个原则:

| 原则 | 含义 |
|------|------|
| **回答 brief 里的所有问题** | 不要漏 |
| **结构清晰**(标题、章节、列表) | 像写文档,不像聊天 |
| **引用来源** | 让用户能核查 |
| **不臆造** | 只用 notes 里的事实 |

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 报告阶段的最大坑:**LLM「**填补 notes 没说的内容**」**

Notes 里可能没提到某个细节(因为 supervisor 没派 agent 研究)。**LLM 写报告时,可能会用「**通识知识**」补**。

**后果**:

- 报告里有 **没经过研究验证的内容**
- 用户以为是研究 findings,**实际上是 LLM 编的**
- **最严重的 hallucination 形式**(混在真内容里,难以发现)

**对治**:prompt 里强烈强调 **「**只用 notes 里出现过的事实**」**,「**如果 notes 没说,直接写 "未研究"**」。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比 — 整个 Pipeline 像什么?

</div>

### 🅰️ 类比:**咨询公司的项目流程**

| 我们的阶段 | 咨询公司 | 类比 |
|----------|---------|------|
| **Scoping** | **Intake meeting** + Project charter | 咨询师跟客户对齐需求 |
| **Supervisor 派单** | **Partner 分配工作** | Partner 把 deck 不同部分分给 senior consultants |
| **Sub-agent 研究** | **Senior consultants 各做一块** | 每人深挖自己负责的子话题 |
| **Compressed research** | **每人交一份 内部 deck** | 精炼总结,不是原始 trail |
| **Report Generation** | **Partner 综合成 final deliverable** | 整合各人的 deck,给客户最终报告 |

→ deep research 系统 = **AI 版的咨询公司流水线**。

### 🅱️ 类比:**操作系统的进程模型**

| 我们的概念 | OS 类比 |
|----------|---------|
| **Supervisor** | **主进程**(分配任务) |
| **Sub-agents** | **子进程**(各自有隔离内存空间) |
| **Compressed research 回传** | **进程间通信**(IPC) |
| **`recursion_limit`** | **栈深度限制**(防止无限递归) |
| **`AgentState`(内部)vs `AgentInputState`(外部)** | **内核态 vs 用户态接口** |

→ LangGraph 的设计 **深受系统编程经典模式启发**。

### 🆚 对比:**单 LLM 调用 vs Deep Research Pipeline**

| 维度 | 直接问 ChatGPT | Deep Research Pipeline |
|------|---------------|----------------------|
| **信息来源** | ❌ 训练数据(可能过时) | ✅ **实时搜索 + 多源验证** |
| **深度** | 🟡 表面 | ✅ **多轮深挖** |
| **可信度** | ❌ 无法核查 | ✅ **每条都有出处** |
| **结构化** | 🟡 一段大文本 | ✅ **正式报告格式** |
| **耗时** | ⚡ 秒级 | 🐢 几分钟 |
| **成本** | 💰 1 次调用 | 💰💰💰 几十次 LLM + 搜索 |

**结论**:**简单问题用 ChatGPT,深度研究用 pipeline**。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 部署 + 生产化清单

</div>

**整个课程产出的代码** 已经是 **deploy-ready** 的:

```
deep_research_from_scratch/
├── src/deep_research_from_scratch/
│   ├── state.py                  # State 定义
│   ├── research_agent_scope.py   # Scoping(L1)
│   ├── research_agent.py         # Single agent(L2)
│   ├── research_agent_mcp.py     # MCP agent(L3)
│   ├── multi_agent_supervisor.py # Supervisor(L4)
│   ├── research_agent_full.py    # 完整系统(L5)
│   ├── prompts.py                # Prompt 模板
│   └── ...
├── notebooks/                    # 学习用 notebook
├── pyproject.toml                # 依赖
└── langgraph.json                # LangGraph 部署配置
```

**启动方法**:

```bash
uvx --refresh --from "langgraph-cli[inmem]" --with-editable . --python 3.11 langgraph dev --allow-blocking
```

在 LangGraph Studio 下拉里选 `research_agent_full`。

### ✅ 生产化检查清单

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

上生产前必查:

- [ ] **recursion_limit** 设到 50+(单 query 复杂度评估)
- [ ] **max_tokens** 显式设大(压缩节点 + 报告节点都需要)
- [ ] **search 工具的 max_results** 用 `InjectedToolArg` 锁死(防 LLM 乱填)
- [ ] **Tavily API key** 走环境变量,**不要硬编码**
- [ ] **LangSmith tracing** 打开(LANGSMITH_TRACING=true)
- [ ] **错误处理**:tool 失败时 ToolMessage 返回错误说明(让 LLM 决定重试 / 换工具)
- [ ] **超时控制**:整个 pipeline 加总超时(避免一次研究跑 1 小时)
- [ ] **token 预算**:加 LangSmith 监控,跑 N 次取均值,**算清单次成本**
- [ ] **eval suite**:scoping / single agent / supervisor 都有测试用例
- [ ] **回归测试**:每次改 prompt 自动跑 eval

</div>

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 5 个最容易踩的坑**

1. **❌ 用默认 `recursion_limit=25`** → 复杂 query 跑一半被 cap → 错误信息「**Recursion limit reached**」

2. **❌ 报告 prompt 没强调「**只用 notes 里的事实**」** → LLM 用通识补内容 → **隐蔽 hallucination**

3. **❌ 子图之间 state 字段冲突** → 比如两个子图都改 `messages`,reducer 没设对 → 信息丢

4. **❌ 没设单次 query 超时** → 一次研究跑 30 分钟,用户以为卡死

5. **❌ 上生产前没跑 eval** → 改了某个 prompt 不知道有没有变差,**线上才发现回归**

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Deep Research System = 「**Scoping → Research(via Supervisor → Sub-agents) → Report**」的 4 阶段 pipeline**:

- **递归组合性**:子图作 node 嵌入,**LangGraph 三件套就够搭任意复杂系统**
- **`recursion_limit` 必须调大**(默认 25 不够复杂 pipeline)
- **每一步都要测**(eval 是 ambient / agentic 系统的生死线)
- **报告阶段防 hallucination**("只用 notes 里的事实")

### 🎯 整门课走完,你掌握了什么

| 能力 | 来自 |
|------|------|
| ✅ 用 `with_structured_output` 让 LLM 做结构化决策 | L1 |
| ✅ 用 `Command(goto, update)` 简化路由 | L1 |
| ✅ 接口隔离 `AgentInputState(MessagesState): pass` | L1 |
| ✅ 防 agent 失控的 3 个 prompt 技巧 | L2 |
| ✅ 双层 context engineering(网页摘要 + 研究压缩) | L2 |
| ✅ `InjectedToolArg` 隐藏 LLM 不该管的参数 | L2 |
| ✅ MCP 集成(标准化工具协议) | L3 |
| ✅ Multi-agent + context isolation | L4 |
| ✅ 「偏好单 agent」的反直觉准则 | L4 |
| ✅ 递归组合 / 子图作 node | L5 |
| ✅ recursion_limit / 报告防 hallucination | L5 |
| ✅ LangSmith eval 贯穿全程 | L1-L5 |

### 🚀 下一步可以做的事

- **迁移这套架构** 到你的真实场景(竞品分析 / 投资研究 / 法律检索 / 技术调研)
- **接 MCP server**(Linear / Slack / 公司内部 KB)替代 Tavily
- **部署到 LangGraph Platform**(生产级 Postgres checkpointer + 监控)
- **接 Agent Inbox / 自定义 UI**(让 HITL 进来,关键步骤可审核)

📎 配套阅读:[`5_full_agent.ipynb` 主课](./5_full_agent.ipynb) · 前置 [`4_z`](./4_z_⭐️_本节精华_Supervisor.ipynb) · 全程精华系列 [`1_z`](./1_z_⭐️_本节精华_Scoping.ipynb) [`2_z`](./2_z_⭐️_本节精华_ResearchAgent.ipynb) [`3_z`](./3_z_⭐️_本节精华_MCP.ipynb) [`4_z`](./4_z_⭐️_本节精华_Supervisor.ipynb)

</div>